# 🏥 City Hospital — Live ML Training Dashboard
Breast Cancer Wisconsin (Diagnostic) · PyTorch · Google Colab

**How to use:** run the cells from top to bottom (Runtime → Run all, or Shift+Enter one at a time).
Read the markdown above each section first, then the output below it.
Educational project only. Never use it to diagnose real patients.

| Part | What it does |
|---|---|
| 1 | Setup and imports |
| 2 | Load the data |
| 3 | EDA (exploratory data analysis) |
| 4 | Preprocessing (clean, encode, split, scale) |
| 5 | Model and training engine |
| 6 | Six experiments |
| 7 | Compare experiments |
| 8 | Live training dashboard |
| 9 | Final model, test evaluation (once) |
| 10 | Prediction demo |
| 11 | Save and export |

## 1. Setup and imports

In [ ]:
import os, copy, json, warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, ConfusionMatrixDisplay)
import joblib
from IPython.display import display, clear_output

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# One colour per class, used in every chart
PAL = {"B": "#2a9d8f", "M": "#e63946"}   # B = benign (green), M = malignant (red)
print("Torch version:", torch.__version__)

## 2. Load the data
The cell tries three ways, in order:
1. a `data.csv` you already uploaded to Colab (folder icon on the left),
2. downloading it from Kaggle automatically,
3. a file-upload button (download `data.csv` from Kaggle yourself first).

In [ ]:
DATA_PATH = None

if os.path.exists("data.csv"):
    DATA_PATH = "data.csv"

if DATA_PATH is None:
    try:
        try:
            import kagglehub
        except ImportError:
            import subprocess, sys
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
            import kagglehub
        folder = kagglehub.dataset_download("uciml/breast-cancer-wisconsin-data")
        DATA_PATH = os.path.join(folder, "data.csv")
    except Exception as e:
        print("Automatic Kaggle download failed:", e)

if DATA_PATH is None or not os.path.exists(DATA_PATH):
    from google.colab import files
    print("Please choose data.csv")
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH)
print("Shape :", df.shape)

## 3. EDA — exploratory data analysis
Goal: understand the data **before** modelling. Questions we answer:
how big is it, is anything missing, are the classes balanced, which features separate benign from malignant?

### 3.1 First look

In [ ]:
display(df.head())
print()
df.info()

### 3.2 Missing values and duplicates

In [ ]:
missing = df.isnull().sum()
print("Columns with missing values:")
print(missing[missing > 0] if (missing > 0).any() else "none")
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate ids :", df["id"].duplicated().sum() if "id" in df.columns else "n/a")
# NOTE: some copies of this dataset have an empty column called "Unnamed: 32". It is junk and gets dropped later.

### 3.3 Class balance (the target column)

In [ ]:
counts = df["diagnosis"].value_counts().reindex(["B", "M"])
pct = (counts / len(df) * 100).round(1)
print(pd.DataFrame({"count": counts, "percent": pct}))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].bar(["Benign", "Malignant"], counts.values, color=[PAL["B"], PAL["M"]])
for i, v in enumerate(counts.values):
    ax[0].text(i, v + 5, str(v), ha="center", fontweight="bold")
ax[0].set_title("Number of cases")
ax[1].pie(counts.values, labels=["Benign", "Malignant"], autopct="%1.1f%%",
          colors=[PAL["B"], PAL["M"]], startangle=90)
ax[1].set_title("Share of cases")
plt.tight_layout(); plt.show()

### 3.4 Feature groups
The dataset has 10 measurements, each in three versions: `_mean`, `_se` (standard error) and `_worst`.

In [ ]:
target_num = df["diagnosis"].map({"M": 1, "B": 0})          # M = 1 (positive class), B = 0
feature_cols = [c for c in df.columns if c not in ["id", "diagnosis", "Unnamed: 32"]]
mean_cols  = [c for c in feature_cols if c.endswith("_mean")]
se_cols    = [c for c in feature_cols if c.endswith("_se")]
worst_cols = [c for c in feature_cols if c.endswith("_worst")]
print(f"{len(feature_cols)} features -> {len(mean_cols)} mean, {len(se_cols)} se, {len(worst_cols)} worst")
display(df[mean_cols].describe().T.round(3))

### 3.5 Average value per class

In [ ]:
by_class = df.groupby("diagnosis")[mean_cols].mean().T
by_class["M / B ratio"] = (by_class["M"] / by_class["B"]).round(2)
display(by_class.round(3).sort_values("M / B ratio", ascending=False))

### 3.6 Distributions: benign vs malignant

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for ax, col in zip(axes.ravel(), mean_cols):
    sns.histplot(data=df, x=col, hue="diagnosis", kde=True, palette=PAL,
                 element="step", stat="density", common_norm=False, ax=ax)
    ax.set_title(col)
plt.suptitle("Feature distributions by diagnosis (overlap = hard to separate)", y=1.02, fontsize=14)
plt.tight_layout(); plt.show()

### 3.7 Box plots and outliers

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for ax, col in zip(axes.ravel(), mean_cols):
    sns.boxplot(data=df, x="diagnosis", y=col, order=["B", "M"],
                palette=[PAL["B"], PAL["M"]], ax=ax)
    ax.set_title(col)
plt.tight_layout(); plt.show()

# Outlier count per feature with the IQR rule
q1, q3 = df[feature_cols].quantile(0.25), df[feature_cols].quantile(0.75)
iqr = q3 - q1
outliers = ((df[feature_cols] < q1 - 1.5 * iqr) | (df[feature_cols] > q3 + 1.5 * iqr)).sum()
print("Features with most outliers (IQR rule):")
print(outliers.sort_values(ascending=False).head(10))

### 3.8 Correlation heatmap

In [ ]:
corr = df[mean_cols].assign(diagnosis=target_num).corr()
plt.figure(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, linewidths=0.5)
plt.title("Correlation of the 10 mean features + diagnosis (M = 1)")
plt.tight_layout(); plt.show()

### 3.9 Which features relate most to the diagnosis?

In [ ]:
corr_target = df[feature_cols].corrwith(target_num).sort_values()
plt.figure(figsize=(8, 9))
colors = [PAL["B"] if v < 0 else PAL["M"] for v in corr_target.values]
plt.barh(corr_target.index, corr_target.values, color=colors)
plt.title("Correlation of every feature with malignant (M = 1)")
plt.xlabel("correlation"); plt.tight_layout(); plt.show()
print("Top 10 by absolute correlation:")
print(corr_target.abs().sort_values(ascending=False).head(10).round(3))

### 3.10 Multicollinearity
Many features measure almost the same thing (radius, perimeter and area are all "size").
Highly correlated features are not wrong for a neural network, but it is good to know they exist.

In [ ]:
cm = df[feature_cols].corr().abs()
upper = cm.where(np.triu(np.ones(cm.shape), k=1).astype(bool))
pairs = upper.stack().sort_values(ascending=False)
print("Number of feature pairs with correlation > 0.9:", (pairs > 0.9).sum())
display(pairs.head(10).round(3).rename("abs correlation").to_frame())

### 3.11 Pair plot of five key features

In [ ]:
key = ["radius_mean", "texture_mean", "concavity_mean", "smoothness_mean", "area_mean"]
g = sns.pairplot(df, vars=key, hue="diagnosis", palette=PAL, corner=True, plot_kws={"alpha": 0.6, "s": 18})
g.fig.suptitle("Pair plot: where do benign and malignant separate?", y=1.02)
plt.show()

### 3.12 Scale differences (why we need scaling)

In [ ]:
ranges = df[feature_cols].agg(["min", "max"]).T
ranges["range"] = ranges["max"] - ranges["min"]
display(ranges.sort_values("range", ascending=False).head(6).round(3))
display(ranges.sort_values("range").head(6).round(5))
print("`area_worst` spans thousands while `fractal_dimension_se` spans fractions of 1.")
print("A neural network trains badly when inputs live on such different scales.")

### 3.13 EDA summary — write your own answers here
- Class balance: ...
- Missing values: ...
- Features that separate the classes best: ...
- Why scaling is needed: ...

## 4. Preprocessing
Order matters: **split first, scale after**, and fit the scaler on the training set only.
Otherwise information from validation/test leaks into training (data leakage).

In [ ]:
# Step 1: drop useless columns, keep the ids aside for the prediction demo
ids = df["id"].copy() if "id" in df.columns else pd.Series(range(len(df)), name="id")
data = df.drop(columns=[c for c in ["id", "Unnamed: 32"] if c in df.columns]).copy()

# Step 2: encode the target.  M = 1 (positive class), B = 0
data["diagnosis"] = data["diagnosis"].map({"M": 1, "B": 0})
assert data.isnull().sum().sum() == 0, "There are still missing values"

X = data.drop(columns="diagnosis")
y = data["diagnosis"]

# Step 3: stratified 60 / 20 / 20 split
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=SEED)

summary = pd.DataFrame({
    "samples":   [len(X_train), len(X_val), len(X_test)],
    "malignant": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
}, index=["train", "validation", "test"])
summary["malignant %"] = (summary["malignant"] / summary["samples"] * 100).round(1)
display(summary)

# Step 4: scaling -> fit on TRAIN only
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

print("\nScaled train mean (should be ~0):", X_train_s.mean().round(4))
print("Scaled train std  (should be ~1):", X_train_s.std().round(4))
print("Scaled val mean   (close to 0, not exactly):", X_val_s.mean().round(4))

# Tensors for both versions: scaled and raw (needed for the "no scaling" experiment)
def to_t(a): return torch.tensor(np.asarray(a), dtype=torch.float32)
TENSORS = {
    True:  dict(Xtr=to_t(X_train_s),        Xva=to_t(X_val_s),        Xte=to_t(X_test_s)),
    False: dict(Xtr=to_t(X_train.values),   Xva=to_t(X_val.values),   Xte=to_t(X_test.values)),
}
ytr_t, yva_t, yte_t = to_t(y_train), to_t(y_val), to_t(y_test)
N_FEATURES = X_train.shape[1]
print("\nFeatures:", N_FEATURES)

## 5. Model and training engine

**Model:** `Linear(30, hidden) → ReLU → Dropout → Linear(hidden, 1)`; it outputs a *logit* (a raw score).
`BCEWithLogitsLoss` applies the sigmoid internally, so the model itself does not.

**One training epoch:**
1. forward pass → predictions
2. loss → how wrong we are
3. `loss.backward()` → gradients
4. `optimizer.step()` → update weights

After every epoch we measure train and validation loss (in `eval()` mode, no dropout) so the two curves are comparable.

In [ ]:
class HospitalNet(nn.Module):
    def __init__(self, n_features, hidden=16, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(1)        # logits, shape (batch,)

criterion = nn.BCEWithLogitsLoss()

def make_cfg(**kw):
    """Default settings, override what you need. patience=None means early stopping is OFF."""
    cfg = dict(hidden=16, dropout=0.3, lr=1e-3, weight_decay=0.05,
               patience=20, max_epochs=200, scaling=True)
    cfg.update(kw)
    return cfg

@torch.no_grad()
def loss_on(model, X, y):
    model.eval()
    return criterion(model(X), y).item()

def train_model(cfg, Xtr, ytr, Xva=None, yva=None, fixed_epochs=None,
                batch_size=32, progress_cb=None):
    """Train one model. Returns (model, history, best_epoch)."""
    torch.manual_seed(SEED)
    model = HospitalNet(Xtr.shape[1], cfg["hidden"], cfg["dropout"])
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True,
                        generator=torch.Generator().manual_seed(SEED))
    max_epochs = fixed_epochs or cfg["max_epochs"]
    history = {"train_loss": [], "val_loss": []}
    best_val, best_epoch, best_state, wait = float("inf"), 0, None, 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        history["train_loss"].append(loss_on(model, Xtr, ytr))
        if Xva is not None:
            v = loss_on(model, Xva, yva)
            history["val_loss"].append(v)
            if v < best_val - 1e-6:                       # validation improved
                best_val, best_epoch, wait = v, epoch, 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                wait += 1
                if cfg["patience"] is not None and wait >= cfg["patience"]:
                    break                                   # early stopping
        if progress_cb:
            progress_cb(epoch, history, max_epochs)

    # With early stopping we go back to the best epoch's weights
    if Xva is not None and cfg["patience"] is not None and best_state is not None:
        model.load_state_dict(best_state)
    if Xva is None:
        best_epoch = max_epochs
    return model, history, best_epoch

@torch.no_grad()
def evaluate(model, X, y, threshold=0.5):
    """Loss + classification metrics. Malignant (1) is the positive class."""
    model.eval()
    logits = model(X)
    probs = torch.sigmoid(logits).numpy()
    preds = (probs >= threshold).astype(int)
    yt = y.numpy().astype(int)
    tn, fp, fn, tp = confusion_matrix(yt, preds, labels=[0, 1]).ravel()
    return dict(
        loss=criterion(logits, y).item(),
        accuracy=accuracy_score(yt, preds),
        precision=precision_score(yt, preds, zero_division=0),
        recall=recall_score(yt, preds, zero_division=0),
        f1=f1_score(yt, preds, zero_division=0),
        tp=int(tp), fn=int(fn), fp=int(fp), tn=int(tn),
        probs=probs, preds=preds,
    )

def diagnose(history):
    """Underfitting / Overfitting / Healthy from the loss curves."""
    tr, va = history["train_loss"], history["val_loss"]
    if tr[-1] > 0.30:
        return "Underfitting"
    # FINAL validation loss vs the best loss from an EARLIER epoch (the last epoch is excluded)
    if len(va) > 1 and va[-1] > 1.30 * min(va[:-1]):
        return "Overfitting"
    return "Healthy"

print("Engine ready.")

## 6. Experiments
Every run is logged to `RESULTS` (a list that becomes a DataFrame) together with its loss curves.
Each experiment changes as few things as possible compared with the baseline.

> Your plan lists experiments 1 and 5 with identical settings. To make experiment 5 informative,
> it uses **8 neurons** instead of 16 (a genuinely smaller model).

In [ ]:
RESULTS, HISTORIES, CONFIGS = [], {}, {}

def run_experiment(name, cfg, progress_cb=None):
    T = TENSORS[cfg["scaling"]]
    model, hist, best_ep = train_model(cfg, T["Xtr"], ytr_t, T["Xva"], yva_t, progress_cb=progress_cb)
    tr = evaluate(model, T["Xtr"], ytr_t)
    va = evaluate(model, T["Xva"], yva_t)
    row = {
        "Experiment ID": name,
        "Date": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "Hidden Neurons": cfg["hidden"],
        "Dropout": cfg["dropout"],
        "Learning Rate": cfg["lr"],
        "Weight Decay": cfg["weight_decay"],
        "Patience": cfg["patience"] if cfg["patience"] is not None else "off",
        "Scaling": cfg["scaling"],
        "Epochs Run": len(hist["train_loss"]),
        "Best Epoch": best_ep,
        "Training Loss": round(tr["loss"], 4),
        "Validation Loss": round(va["loss"], 4),
        "Accuracy": round(va["accuracy"], 4),
        "Precision": round(va["precision"], 4),
        "Recall": round(va["recall"], 4),
        "F1": round(va["f1"], 4),
        "Cancers Found": va["tp"],
        "Cancers Missed": va["fn"],
        "False Alarms": va["fp"],
        "Diagnosis": diagnose(hist),
    }
    RESULTS.append(row); HISTORIES[name] = hist; CONFIGS[name] = cfg
    return row, model, hist, va

EXPERIMENTS = [
    ("1. Baseline",       make_cfg()),
    ("2. Overfitting",    make_cfg(hidden=256, dropout=0.0, weight_decay=0.0, patience=None)),
    ("3. Regularization", make_cfg(hidden=256, dropout=0.3, weight_decay=0.05, patience=None)),
    ("4. Early stopping", make_cfg(hidden=256, dropout=0.3, weight_decay=0.05, patience=20)),
    ("5. Small model",    make_cfg(hidden=8)),
    ("6. No scaling",     make_cfg(scaling=False)),
]

for name, cfg in EXPERIMENTS:
    row, *_ = run_experiment(name, cfg)
    print(f"{name:<20} epochs={row['Epochs Run']:>3}  best={row['Best Epoch']:>3}  "
          f"train={row['Training Loss']:.3f}  val={row['Validation Loss']:.3f}  "
          f"recall={row['Recall']:.3f}  -> {row['Diagnosis']}")

## 7. Compare experiments

In [ ]:
results_df = pd.DataFrame(RESULTS)
display(results_df)

In [ ]:
# Loss curves of every experiment
names = [n for n, _ in EXPERIMENTS]
fig, axes = plt.subplots(2, 3, figsize=(17, 8), sharey=False)
for ax, n in zip(axes.ravel(), names):
    h = HISTORIES[n]
    ax.plot(h["train_loss"], label="Train", color="#264653", lw=2)
    ax.plot(h["val_loss"], label="Validation", color="#e76f51", lw=2)
    best = int(np.argmin(h["val_loss"]))
    ax.axvline(best, color="grey", ls="--", lw=1)
    ax.set_title(f"{n}  [{results_df.loc[results_df['Experiment ID']==n, 'Diagnosis'].iloc[0]}]")
    ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend()
plt.suptitle("Training vs validation loss (dashed line = best validation epoch)", fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# Metric comparison: accuracy alone hides missed cancers, so look at recall too
plot_df = results_df.set_index("Experiment ID")[["Accuracy", "Precision", "Recall", "F1"]]
plot_df.plot(kind="bar", figsize=(12, 5), colormap="viridis", edgecolor="white")
plt.ylim(0.5, 1.02); plt.title("Validation metrics per experiment"); plt.xticks(rotation=25, ha="right")
plt.tight_layout(); plt.show()

missed = results_df.set_index("Experiment ID")[["Cancers Found", "Cancers Missed", "False Alarms"]]
display(missed)

**Questions to answer in your report** (use the table and charts above):
1. Which experiment has the biggest gap between training and validation loss?
2. Did dropout + weight decay (exp 3) shrink that gap compared with exp 2?
3. Did early stopping (exp 4) pick a better epoch than running to the end (exp 3)?
4. Did the smaller model (exp 5) generalise better than the baseline?
5. What did removing scaling (exp 6) do to the loss curves?
6. Which experiment missed the fewest cancers? Is that the same one with the best accuracy?

*Tip: if experiment 2 does not overfit strongly on your run, try a larger learning rate (e.g. 0.01) in the dashboard below with dropout 0, weight decay 0 and early stopping off.*

## 8. Live training dashboard
Set the controls, press **Train Model**, and watch the loss curves update live.
Every run is added to the experiment history table.

In [ ]:
import ipywidgets as widgets

sty = {"description_width": "150px"}
lay = widgets.Layout(width="480px")
w_hidden = widgets.Dropdown(options=[8, 16, 32, 64, 128, 256], value=16, description="Hidden neurons", style=sty, layout=lay)
w_drop   = widgets.FloatSlider(value=0.30, min=0.0, max=0.8, step=0.05, description="Dropout", style=sty, layout=lay)
w_lr     = widgets.FloatLogSlider(value=0.001, base=10, min=-4, max=-1, step=0.25, description="Learning rate", readout_format=".4f", style=sty, layout=lay)
w_wd     = widgets.FloatSlider(value=0.05, min=0.0, max=0.2, step=0.005, description="Weight decay", readout_format=".3f", style=sty, layout=lay)
w_pat    = widgets.IntSlider(value=20, min=0, max=50, description="Patience (0 = off)", style=sty, layout=lay)
w_ep     = widgets.IntSlider(value=200, min=10, max=500, step=10, description="Maximum epochs", style=sty, layout=lay)
w_scale  = widgets.Checkbox(value=True, description="Feature scaling", style=sty)
btn      = widgets.Button(description="Train Model", button_style="success", icon="play")
out      = widgets.Output()
RUN = {"n": 0}

def live_plot(epoch, history, max_epochs):
    if epoch != 1 and epoch % 5 != 0:
        return
    clear_output(wait=True)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.plot(history["train_loss"], label="Train", color="#264653", lw=2)
    ax.plot(history["val_loss"], label="Validation", color="#e76f51", lw=2)
    ax.set_title(f"Training... epoch {epoch}/{max_epochs}")
    ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend()
    plt.show()

def on_train(_):
    RUN["n"] += 1
    cfg = make_cfg(hidden=w_hidden.value, dropout=w_drop.value, lr=float(w_lr.value),
                   weight_decay=w_wd.value, patience=(w_pat.value or None),
                   max_epochs=w_ep.value, scaling=w_scale.value)
    btn.disabled = True
    with out:
        clear_output()
        row, model, hist, va = run_experiment(f"Dash-{RUN['n']}", cfg, progress_cb=live_plot)
        clear_output(wait=True)
        total_cancers = va["tp"] + va["fn"]
        print("🏥 City Hospital — Training Lab")
        print("-" * 44)
        print(f"Training loss    : {row['Training Loss']:.3f}")
        print(f"Validation loss  : {row['Validation Loss']:.3f}")
        print(f"Accuracy         : {row['Accuracy']*100:.1f}%")
        print(f"Recall           : {row['Recall']*100:.1f}%")
        print(f"Cancers detected : {va['tp']} / {total_cancers}")
        print(f"False alarms     : {va['fp']}")
        print(f"Epochs run / best: {row['Epochs Run']} / {row['Best Epoch']}")
        print(f"Diagnosis        : {row['Diagnosis']}")
        fig, ax = plt.subplots(figsize=(7, 3.5))
        ax.plot(hist["train_loss"], label="Train", color="#264653", lw=2)
        ax.plot(hist["val_loss"], label="Validation", color="#e76f51", lw=2)
        ax.axvline(row["Best Epoch"] - 1, color="grey", ls="--", lw=1)
        ax.set_title(f"Dash-{RUN['n']}: {row['Diagnosis']}")
        ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend()
        plt.show()
    btn.disabled = False

btn.on_click(on_train)
display(widgets.VBox([w_hidden, w_drop, w_lr, w_wd, w_pat, w_ep, w_scale, btn, out]))

In [ ]:
# Run this any time to see ALL experiments (the six above + every dashboard run)
display(pd.DataFrame(RESULTS))

## 9. Final model and test evaluation
Rules we follow:
1. Pick the best configuration using **validation** results only.
2. Freeze its settings.
3. Retrain on **train + validation** (the development set), with the scaler fitted on that combined set.
4. Train for the best epoch count found earlier (no validation set is left to stop on).
5. Evaluate on the **test set once**. No tuning after this.

In [ ]:
res = pd.DataFrame(RESULTS)
# Prefer runs diagnosed "Healthy" (an overfit/underfit run is not a model we want to ship),
# then rank by validation loss, then by fewest missed cancers.
candidates = res[res["Diagnosis"] == "Healthy"]
if candidates.empty:
    candidates = res
best_row = candidates.sort_values(["Validation Loss", "Cancers Missed"]).iloc[0]
best_name = best_row["Experiment ID"]
best_cfg = CONFIGS[best_name]
print("Best configuration by validation loss:", best_name)
print(json.dumps(best_cfg, indent=2))
display(best_row.to_frame("value"))

In [ ]:
# Combined development set (train + validation), scaler fitted on it
if best_cfg["scaling"]:
    final_scaler = StandardScaler().fit(X_dev)
    Xdev_t = to_t(final_scaler.transform(X_dev))
    Xte_t  = to_t(final_scaler.transform(X_test))
else:
    final_scaler = None
    Xdev_t, Xte_t = to_t(X_dev.values), to_t(X_test.values)
ydev_t = to_t(y_dev)

# With early stopping the best epoch is the right length; without it the run used all its epochs.
if best_cfg["patience"] is None:
    final_epochs = int(best_row["Epochs Run"])
else:
    final_epochs = max(int(best_row["Best Epoch"]), 1)
final_model, final_hist, _ = train_model(best_cfg, Xdev_t, ydev_t, fixed_epochs=final_epochs)

# ---- the one and only look at the test set ----
test = evaluate(final_model, Xte_t, yte_t)
print(f"Retrained on {len(X_dev)} samples for {final_epochs} epochs.\n")
print("TEST RESULTS")
print(f"  Accuracy : {test['accuracy']*100:.1f}%")
print(f"  Precision: {test['precision']*100:.1f}%")
print(f"  Recall   : {test['recall']*100:.1f}%")
print(f"  F1       : {test['f1']*100:.1f}%")
print(f"  Cancers detected: {test['tp']} / {test['tp'] + test['fn']}")
print(f"  Cancers missed  : {test['fn']}   False alarms: {test['fp']}")

In [ ]:
cm_test = confusion_matrix(y_test, test["preds"], labels=[0, 1])
ConfusionMatrixDisplay(cm_test, display_labels=["Benign", "Malignant"]).plot(cmap="Blues", values_format="d")
plt.title("Confusion matrix — test set"); plt.show()
print(classification_report(y_test, test["preds"], target_names=["Benign", "Malignant"], digits=3))

## 10. Prediction demo (educational only)

In [ ]:
test_ids = ids.loc[X_test.index].values
test_df = pd.DataFrame({
    "Sample ID": test_ids,
    "Actual": np.where(y_test.values == 1, "Malignant", "Benign"),
    "Predicted": np.where(test["preds"] == 1, "Malignant", "Benign"),
    "P(malignant) %": np.round(test["probs"].astype(float) * 100, 1),
    "P(benign) %": np.round((1 - test["probs"].astype(float)) * 100, 1),
})
test_df["Correct"] = np.where(test_df["Actual"] == test_df["Predicted"], "Yes", "NO")

def show_sample(sample_id):
    r = test_df[test_df["Sample ID"] == sample_id]
    if r.empty:
        print("Sample ID is not in the test set."); return
    r = r.iloc[0]
    print(f"Sample ID           : {r['Sample ID']}")
    print(f"Actual diagnosis    : {r['Actual']}")
    print(f"Predicted diagnosis : {r['Predicted']}")
    print(f"Malignant prob.     : {r['P(malignant) %']}%")
    print(f"Benign prob.        : {r['P(benign) %']}%")
    print(f"Prediction result   : {'Correct' if r['Correct']=='Yes' else 'WRONG'}")
    print("\nEducational demo only, never for real diagnosis.")

show_sample(test_df.sample(1, random_state=1)["Sample ID"].iloc[0])

print("\nWrong predictions on the test set:")
display(test_df[test_df["Correct"] == "NO"])

In [ ]:
# Pick any test sample with a dropdown
dd = widgets.Dropdown(options=list(test_df["Sample ID"]), description="Sample ID")
pout = widgets.Output()
def _chg(change):
    with pout:
        clear_output(); show_sample(change["new"])
dd.observe(_chg, names="value")
display(dd, pout)

## 11. Save and export

In [ ]:
os.makedirs("outputs", exist_ok=True)

# model + scaler + settings
torch.save({"state_dict": final_model.state_dict(),
            "n_features": N_FEATURES,
            "hidden": best_cfg["hidden"], "dropout": best_cfg["dropout"],
            "feature_names": list(X.columns),
            "scaling": best_cfg["scaling"]}, "outputs/hospital_model.pt")
if final_scaler is not None:
    joblib.dump(final_scaler, "outputs/scaler.pkl")
with open("outputs/best_config.json", "w") as f:
    json.dump({"experiment": best_name, "config": best_cfg, "final_epochs": final_epochs}, f, indent=2)

# experiment history + final test metrics + predictions
pd.DataFrame(RESULTS).to_csv("outputs/experiment_history.csv", index=False)
pd.DataFrame([{k: v for k, v in test.items() if k not in ("probs", "preds")}]).to_csv("outputs/final_test_metrics.csv", index=False)
test_df.to_csv("outputs/test_predictions.csv", index=False)

# loss curves picture
fig, axes = plt.subplots(2, 3, figsize=(17, 8))
for ax, n in zip(axes.ravel(), names):
    ax.plot(HISTORIES[n]["train_loss"], label="Train"); ax.plot(HISTORIES[n]["val_loss"], label="Validation")
    ax.set_title(n); ax.legend()
plt.tight_layout(); plt.savefig("outputs/loss_curves.png", dpi=150); plt.close()
cm_fig = ConfusionMatrixDisplay(cm_test, display_labels=["Benign", "Malignant"]).plot(cmap="Blues", values_format="d")
plt.title("Confusion matrix — test set"); plt.savefig("outputs/confusion_matrix_test.png", dpi=150); plt.close()

print("Saved files:")
for f in sorted(os.listdir("outputs")):
    print("  outputs/" + f)

In [ ]:
# Download everything as one zip (Colab). Outside Colab this simply prints a note.
import shutil
shutil.make_archive("hospital_outputs", "zip", "outputs")
try:
    from google.colab import files
    files.download("hospital_outputs.zip")
except ImportError:
    print("Not running in Colab. The zip is in the current folder: hospital_outputs.zip")

## Next steps
1. Write your EDA summary (section 3.13) and your answers to the six questions (section 7).
2. Upload the notebook and the `outputs/` folder to a new GitHub repository with a README.
3. Make 5–8 slides: problem, data, method, experiments, results, limits.
4. Then we start Project 2 (credit card fraud).

**Limitations to mention:** small dataset (569 rows), one random split, results change slightly with the seed, not for clinical use.